# KEK Power Competitiveness — End-to-End Scorecard

Runs `build_scorecard()` from `src.model.basic_model` using the real precomputed pipeline tables.

**Data inputs (all from `outputs/data/processed/`)**:
| Table | Content |
|-------|--------|
| `dim_kek.csv` | KEK master list (identity, grid region, reliability requirement) |
| `fct_kek_resource.csv` | PVOUT at centroid + best-within-50km (from Global Solar Atlas GeoTIFF) |
| `fct_ruptl_pipeline.csv` | PLTS (solar) capacity additions by grid region & year (RE Base + ARED) |
| `fct_grid_cost_proxy.csv` | Dashboard reference rate per grid region (I-4/TT tariff, $63.08/MWh) |
| `dim_tech_cost.csv` | TECH006 CAPEX/FOM/lifetime from ESDM technology catalogue |

**Demand**: Placeholder 2030 demand is used (no `fct_kek_demand.csv` yet). Manufacturing KEKs = 500 GWh/yr, Service = 200 GWh/yr. Replace when real demand data is available.

**Model**: `src.model.basic_model.build_scorecard()` — pure Python, no Dash dependency.

In [ ]:
import sys
from pathlib import Path

# Add repo root to sys.path so src.model imports work from notebooks/
REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

from src.model.basic_model import (
    build_scorecard,
    geas_policy_allocation,
    action_flags,
)
from src.assumptions import (
    TECH006_CAPEX_USD_PER_KW,
    TECH006_FOM_USD_PER_KW_YR,
    TECH006_LIFETIME_YR,
    BASE_WACC_DECIMAL,
    TARIFF_I4_USD_MWH,
)

PROCESSED = REPO_ROOT / "outputs" / "data" / "processed"
print(f"Repo root : {REPO_ROOT}")
print(f"Processed : {PROCESSED}")

## 1. Load real data tables

In [ ]:
dim_kek       = pd.read_csv(PROCESSED / "dim_kek.csv")
fct_resource  = pd.read_csv(PROCESSED / "fct_kek_resource.csv")
fct_ruptl     = pd.read_csv(PROCESSED / "fct_ruptl_pipeline.csv")
fct_grid_cost = pd.read_csv(PROCESSED / "fct_grid_cost_proxy.csv")
dim_tech_cost = pd.read_csv(PROCESSED / "dim_tech_cost.csv")

print(f"KEKs: {len(dim_kek)}")
print(f"Resource rows: {len(fct_resource)}")
print(f"RUPTL rows: {len(fct_ruptl)}  (regions: {fct_ruptl.grid_region_id.nunique()})")
dim_kek[["kek_id","kek_name","province","grid_region_id","reliability_req"]].head(10)

## 2. Demand placeholder (replace when fct_kek_demand.csv is available)

Until scraper-derived tenant demand data is available, we assign:
- **Manufacturing KEKs** (reliability_req ≥ 0.8): 500,000 MWh/yr
- **Service/Mixed KEKs** (reliability_req < 0.8): 200,000 MWh/yr

These are rough order-of-magnitude proxies based on typical Indonesian industrial park load profiles.

In [ ]:
DEMAND_MANUFACTURING_MWH = 500_000
DEMAND_SERVICE_MWH = 200_000
TARGET_YEAR = 2030

fct_demand = dim_kek[["kek_id","reliability_req"]].copy()
fct_demand["demand_mwh"] = fct_demand["reliability_req"].apply(
    lambda r: DEMAND_MANUFACTURING_MWH if r >= 0.8 else DEMAND_SERVICE_MWH
)
fct_demand["year"] = TARGET_YEAR
fct_demand = fct_demand[["kek_id","year","demand_mwh"]]

print("[PLACEHOLDER] 2030 demand distribution:")
print(fct_demand["demand_mwh"].value_counts().to_string())
print(f"  Total: {fct_demand.demand_mwh.sum():,.0f} MWh")

## 3. Tech cost — pull TECH006 from dim_tech_cost (fall back to assumptions.py defaults)

In [ ]:
tech = dim_tech_cost[dim_tech_cost["tech_id"] == "TECH006"]

if len(tech) > 0:
    row = tech.iloc[0]
    capex  = float(row["capex_usd_per_kw"])
    fom    = float(row["fom_usd_per_kw_yr"])
    life   = int(row["lifetime_yr"])
    is_prov = bool(row.get("is_provisional", True))
    print(f"TECH006 from dim_tech_cost — CAPEX: ${capex}/kW  FOM: ${fom}/kW/yr  Lifetime: {life}yr  provisional={is_prov}")
else:
    capex, fom, life = TECH006_CAPEX_USD_PER_KW, TECH006_FOM_USD_PER_KW_YR, TECH006_LIFETIME_YR
    is_prov = True
    print(f"TECH006 not found — using assumptions.py defaults: CAPEX=${capex}, FOM=${fom}, life={life}yr")

# Grid reference cost: I-4/TT tariff (uniform nationwide)
grid_cost = fct_grid_cost[fct_grid_cost["grid_region_id"] == "JAVA_BALI"]["dashboard_rate_usd_mwh"].iloc[0]
print(f"Grid reference cost: ${grid_cost:.2f}/MWh ({fct_grid_cost[fct_grid_cost['grid_region_id']=='JAVA_BALI']['dashboard_rate_label'].iloc[0]})")

## 4. Run build_scorecard() — baseline scenario

In [ ]:
scorecard = build_scorecard(
    dim_kek=dim_kek[["kek_id","kek_name","province","grid_region_id","reliability_req"]],
    fct_demand=fct_demand,
    fct_pvout=fct_resource[["kek_id","pvout_centroid","pvout_best_50km"]],
    fct_ruptl=fct_ruptl,
    capex_usd_per_kw=capex,
    fom_usd_per_kw_yr=fom,
    wacc=BASE_WACC_DECIMAL,
    lifetime_yr=life,
    grid_cost_usd_mwh=grid_cost,
    target_year=TARGET_YEAR,
)

print(f"Scorecard rows: {len(scorecard)}")
print(f"WACC used: {BASE_WACC_DECIMAL*100:.0f}%  CAPEX: ${capex}/kW  Grid ref: ${grid_cost}/MWh")

## 5. Scorecard summary table

In [ ]:
display_cols = [
    "kek_name", "province", "grid_region_id",
    "pvout_best_50km", "cf_best_50km", "lcoe_usd_mwh",
    "solar_competitive_gap_pct", "solar_attractive",
    "green_share_geas",
    "solar_now", "grid_first", "firming_needed", "plan_late",
]

summary = scorecard[display_cols].copy()
summary = summary.sort_values("solar_competitive_gap_pct")

# Format for display
summary["pvout_best_50km"] = summary["pvout_best_50km"].map("{:.0f} kWh/kWp".format)
summary["cf_best_50km"]    = summary["cf_best_50km"].map("{:.1%}".format)
summary["lcoe_usd_mwh"]    = summary["lcoe_usd_mwh"].map("${:.1f}".format)
summary["solar_competitive_gap_pct"] = summary["solar_competitive_gap_pct"].map("{:+.1f}%".format)
summary["green_share_geas"] = summary["green_share_geas"].map("{:.0%}".format)

pd.set_option("display.max_rows", 30)
pd.set_option("display.max_colwidth", 35)
summary.rename(columns={
    "kek_name": "KEK", "province": "Province", "grid_region_id": "Grid Region",
    "pvout_best_50km": "PVOUT", "cf_best_50km": "CF", "lcoe_usd_mwh": "LCOE (10% WACC)",
    "solar_competitive_gap_pct": "Gap vs Grid",
    "solar_attractive": "Attractive?",
    "green_share_geas": "GEAS Share",
    "solar_now": "solar_now", "grid_first": "grid_first",
    "firming_needed": "firming", "plan_late": "plan_late",
}, inplace=True)

summary

## 6. Chart — PVOUT vs LCOE competitive gap

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

colors = scorecard["solar_attractive"].map({True: "#2196F3", False: "#FF7043", None: "#9E9E9E"})

sc = ax.scatter(
    scorecard["pvout_best_50km"],
    scorecard["lcoe_usd_mwh"],
    c=colors,
    s=120, alpha=0.85, zorder=3,
)

# Grid reference line
ax.axhline(grid_cost, color="#333", linestyle="--", linewidth=1.5, label=f"Grid ref: ${grid_cost:.0f}/MWh (I-4/TT)")

# PVOUT threshold line
ax.axvline(1550, color="#999", linestyle=":", linewidth=1.2, label="PVOUT threshold: 1,550 kWh/kWp")

# Labels for each KEK
for _, row in scorecard.iterrows():
    ax.annotate(
        row["kek_name"].split()[0],  # first word to keep labels short
        xy=(row["pvout_best_50km"], row["lcoe_usd_mwh"]),
        xytext=(4, 3), textcoords="offset points",
        fontsize=7, color="#444",
    )

ax.set_xlabel("PVOUT best within 50 km (kWh/kWp/yr)", fontsize=11)
ax.set_ylabel("Solar LCOE @ 10% WACC (USD/MWh)", fontsize=11)
ax.set_title("Solar resource vs economics — all KEKs\n(blue = competitive vs I-4/TT grid; red = not yet competitive)", fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.25)

plt.tight_layout()
plt.show()

## 7. Action flag breakdown

In [ ]:
flags = ["solar_now", "grid_first", "firming_needed", "plan_late"]
flag_counts = scorecard[flags].sum().rename("count")

print("=== Action flag counts (baseline, WACC=10%) ===")
for flag, count in flag_counts.items():
    pct = count / len(scorecard) * 100
    print(f"  {flag:20s}: {int(count):2d} / {len(scorecard)} KEKs ({pct:.0f}%)")

print()
print("=== KEKs flagged solar_now ===")
if scorecard["solar_now"].any():
    print(scorecard[scorecard["solar_now"]][["kek_name","province","pvout_best_50km","lcoe_usd_mwh","green_share_geas"]].to_string(index=False))
else:
    print("  None — no KEK meets all three conditions (attractive + grid ready + GEAS ≥30%)")

print()
print("=== KEKs flagged firming_needed ===")
print(scorecard[scorecard["firming_needed"]][["kek_name","province","pvout_best_50km","lcoe_usd_mwh"]].to_string(index=False))

## 8. Policy scenario — GEAS allocation shift

Policy accelerates 20% of post-2030 RUPTL solar into the pre-2030 window, then allocates by `demand × PVOUT` score (vs pure pro-rata in baseline).

In [ ]:
policy_input = scorecard[["kek_id","kek_name","grid_region_id","demand_mwh","pvout_best_50km"]].copy()

policy_result = geas_policy_allocation(
    kek_df=policy_input,
    ruptl_df=fct_ruptl,
    shift_fraction=0.20,
    n_priority_regions=2,
)

# Merge baseline + policy green shares
compare = policy_result[["kek_id","kek_name","green_share_geas_policy"]].merge(
    scorecard[["kek_id","green_share_geas","solar_attractive"]],
    on="kek_id",
)
compare = compare.sort_values("green_share_geas_policy", ascending=False)

fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(compare))
w = 0.38
ax.bar(x - w/2, compare["green_share_geas"], w, label="Baseline (pro-rata)", color="#90CAF9")
ax.bar(x + w/2, compare["green_share_geas_policy"], w, label="Policy (demand×PVOUT)", color="#1565C0")
ax.axhline(0.30, color="#FF5722", linestyle="--", linewidth=1, label="30% solar_now threshold")
ax.set_xticks(x)
ax.set_xticklabels(compare["kek_name"].str.split().str[0], rotation=35, ha="right", fontsize=8)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
ax.set_ylabel("GEAS green share (allocated MWh / 2030 demand)")
ax.set_title("GEAS green share: baseline vs. policy scenario (20% RUPTL acceleration)")
ax.legend(fontsize=9)
ax.grid(True, axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

print(f"\nKEKs crossing 30% threshold under policy (not baseline): "
      f"{((compare.green_share_geas < 0.30) & (compare.green_share_geas_policy >= 0.30)).sum()}")

## 9. WACC sensitivity — how LCOE changes at 8% / 10% / 12%

In [ ]:
from src.model.basic_model import lcoe_solar
from src.assumptions import WACC_VALUES

wacc_rows = []
for _, row in scorecard.iterrows():
    cf = row["cf_best_50km"]
    for w in WACC_VALUES:
        lcoe = lcoe_solar(capex, fom, w/100, life, cf)
        wacc_rows.append({"kek_name": row["kek_name"], "wacc_pct": w, "lcoe": lcoe})

wacc_df = pd.DataFrame(wacc_rows)
pivot = wacc_df.pivot(index="kek_name", columns="wacc_pct", values="lcoe")
pivot.columns = [f"LCOE {w:.0f}% WACC" for w in pivot.columns]
pivot["Gap 10% vs grid"] = (pivot["LCOE 10.0% WACC"] - grid_cost) / grid_cost * 100
pivot = pivot.sort_values("Gap 10% vs grid")

pivot.style \
    .format({
        "LCOE 8.0% WACC": "${:.1f}",
        "LCOE 10.0% WACC": "${:.1f}",
        "LCOE 12.0% WACC": "${:.1f}",
        "Gap 10% vs grid": "{:+.1f}%",
    }) \
    .background_gradient(subset=["Gap 10% vs grid"], cmap="RdYlGn_r", vmin=-20, vmax=20)